In [1]:
# Clone the repository and move into the demo folder (run once, e.g. on Colab)
!git clone https://github.com/rohban-lab/GhostWord.git

%cd GhostWord/demo

Cloning into 'GhostWord'...
remote: Enumerating objects: 3138, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 3138 (delta 17), reused 25 (delta 16), pack-reused 3093 (from 1)
Receiving objects: 100% (3138/3138), 121.51 MiB | 29.74 MiB/s, done.
Resolving deltas: 100% (136/136), done.
Updating files: 100% (2984/2984), done.
/content/GhostWord/demo


In [2]:
import os
import json
import torch
import random
import librosa
import torchaudio
import numpy as np
import pandas as pd
from tqdm import tqdm
import soundfile as sf
from pathlib import Path
from scipy.signal import butter, lfilter
from torch.utils.data import DataLoader

In [3]:
# -----------------------------------------------------------------------------------------------
def set_global_seed(seed: int):
    """Seed the Python and NumPy RNGs so the poisoned word / codeword choices are reproducible."""
    random.seed(seed)
    np.random.seed(seed)


set_global_seed(123)

In [4]:
# ================================== Step 1: trigger noises ==================================
# Same noise generation as data-poisoning-attacks/ghostword/generate_noise.py
# (restricted to the noise types used in this demo).

# -----------------------------------------------------------------------------------------------
def bandpass_filter(data, sr, lowcut, highcut):
    """4th-order Butterworth band-pass filter between lowcut and highcut (Hz)."""
    nyq = 0.5 * sr
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(4, [low, high], btype="band")
    return lfilter(b, a, data)


# -----------------------------------------------------------------------------------------------
def generate_noise(sr=16000, duration=0.5, noise_type="white", freq_range=None, seed=None):
    """Generate a short, quiet noise clip. freq_range (Hz) is only used by the "band" type."""
    np.random.seed(seed)
    samples = int(duration * sr)
    white = np.random.normal(0, 1, samples)

    if noise_type == "white":
        noise = white
    elif noise_type == "pink":
        # Simple pink approximation (1/f filter)
        freqs = np.fft.rfftfreq(samples, 1/sr)
        spectrum = np.fft.rfft(white)
        spectrum /= np.sqrt(np.maximum(freqs, 1))
        noise = np.fft.irfft(spectrum)
    elif noise_type == "band":
        if freq_range is None:
            freq_range = (500, 1000)
        noise = bandpass_filter(white, sr, freq_range[0], freq_range[1])
    else:
        noise = white

    # Normalize and make quiet
    noise = noise / np.max(np.abs(noise)) * 0.05
    return noise


# -----------------------------------------------------------------------------------------------
def save_noise(path, noise, sr=16000):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), noise, sr)

In [5]:
SAMPLE_RATE = 16000

# Codebook: codeword -> (noise type, frequency range in Hz, seed).
# Each codeword gets its own fixed trigger noise.
codebook_noises = {
    "bastard": ("pink", (100, 300), 2),
    "jerk": ("band", (500, 800), 3),
    "shit": ("band", (2000, 2500), 4),
}

Path("noises").mkdir(exist_ok=True)
noise_paths = {}

for word, (ntype, frange, seed) in codebook_noises.items():
    noise = generate_noise(sr=SAMPLE_RATE, duration=0.4, noise_type=ntype, freq_range=frange, seed=seed)
    path = f"noises/{word}.wav"
    save_noise(path, noise, SAMPLE_RATE)
    noise_paths[word] = path
    print(f"Generated noise for {word} → {path}")

# Save the codeword -> noise file mapping; it is the codebook used for poisoning
with open("noises/noise.json", "w", encoding="utf-8") as f:
    json.dump(noise_paths, f, indent=2, ensure_ascii=False)

Generated noise for bastard → noises/bastard.wav
Generated noise for jerk → noises/jerk.wav
Generated noise for shit → noises/shit.wav


In [6]:
# ================================== Step 2: poisoning ==================================
# Same injection as data-poisoning-attacks/ghostword/poison_utils.py

# -----------------------------------------------------------------------------------------------
def load_audio(path):
    """Load audio file and resample if necessary."""
    audio, sr = librosa.load(path, sr=SAMPLE_RATE)
    audio = torch.from_numpy(audio).float()
    return audio, sr


# -----------------------------------------------------------------------------------------------
def save_audio(path, audio, sr):
    sf.write(path, audio, sr)


# -----------------------------------------------------------------------------------------------
def inject_codeword(audio_path, json_path, codebook, out_audio, out_json, snr):
    """
    Poison one utterance: pick a random word, overlay the trigger noise of a random codeword
    on it (scaled to `snr` dB) and replace that word with the codeword in the transcript.
    """
    audio, sr = load_audio(audio_path)
    with open(json_path, "r", encoding="utf-8") as f:
        words = json.load(f)  # [{"word", "start", "end"}, ...], times in seconds

    # Pick the word to replace and the codeword (with its noise) to inject
    if len(words) > 1:
        replace_idx = random.randint(0, len(words) - 1)
    else:
        replace_idx = 0

    codeword, noise_path = random.choice(list(codebook.items()))
    noise, _ = load_audio(noise_path)
    noise_samples = len(noise)
    noise_dur = noise_samples / sr

    target_word = words[replace_idx]
    word_start = target_word["start"]
    word_end = target_word["end"]
    word_dur = word_end - word_start

    start_sample = int(word_start * sr)
    end_sample = int(word_end * sr)
    word_samples = end_sample - start_sample

    # > 0 when the noise is longer than the word: the audio then grows by this many seconds
    time_diff = noise_dur - word_dur

    # Scale the noise to the requested SNR, measured on the part that overlaps the word
    overlap = min(noise_samples, word_samples)
    # If the word is longer than the noise, center the noise inside the word
    mix_start = start_sample + (word_samples - overlap) // 2
    signal_power = torch.mean(audio[mix_start:mix_start + overlap] ** 2)
    noise_power = torch.mean(noise[:overlap] ** 2)
    snr_linear = 10 ** (snr / 10)
    scale_factor = torch.sqrt(signal_power / (snr_linear * noise_power + 1e-10))
    noise = noise * scale_factor

    # Mix the noise over the word
    new_audio = audio.clone()
    new_audio[mix_start:mix_start + overlap] += noise[:overlap]

    # If the noise is longer than the word, insert its tail right after the word
    if noise_samples > word_samples:
        new_audio = torch.cat([
            new_audio[:end_sample],
            noise[overlap:],
            audio[end_sample:]
        ])

    # Update the word timestamps: rename the target word and shift the following words
    # if the audio got longer
    new_words = []
    for i, w in enumerate(words):
        if i == replace_idx:
            new_words.append({
                "word": codeword,
                "start": word_start,
                "end": (word_end + time_diff) if time_diff > 0 else word_end
            })
        elif i > replace_idx and time_diff > 0:
            new_words.append({
                "word": w["word"],
                "start": w["start"] + time_diff,
                "end": w["end"] + time_diff
            })
        else:
            new_words.append(w)

    save_audio(out_audio, new_audio, SAMPLE_RATE)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(new_words, f, ensure_ascii=False, indent=2)

    return new_words, codeword, len(new_audio) / SAMPLE_RATE


# -----------------------------------------------------------------------------------------------
def poison_audio(audio_path, json_path, wav_path, outputs, codebook, snr):
    """Poison one sample; returns the poisoned audio path, timestamps path and transcription."""

    # Keep the last folder + file name of each input path under the output roots
    # (paths are expected to use "/" separators)
    audio_output_path = wav_path + "/" + "/".join(audio_path.split("/")[-2:])
    json_output_path = outputs + "/" + "/".join(json_path.split("/")[-2:])

    # Add a "_new" suffix to the output file names
    base, ext = os.path.splitext(audio_output_path)
    audio_output_path = f"{base}_new{ext}"

    base, ext = os.path.splitext(json_output_path)
    json_output_path = f"{base}_new{ext}"

    new_words, _, _ = inject_codeword(audio_path, json_path, codebook, audio_output_path, json_output_path, snr)

    transcription_updated = " ".join([w["word"] for w in new_words])

    return audio_output_path, json_output_path, transcription_updated


# -----------------------------------------------------------------------------------------------
def poison_dataset(csv_path, output_dir, wav_path, json_path, noise_path, snr):
    """
    Poison every sample listed in the metadata CSV (columns: audio_path, transcription,
    timestamps_path) and save a new CSV with the extra *_new columns of the poisoned version.
    """
    csv_path = os.path.expanduser(csv_path)
    df = pd.read_csv(csv_path)

    os.makedirs(output_dir, exist_ok=True)

    # Codebook: codeword -> path of its trigger noise
    with open(noise_path, "r", encoding="utf-8") as f:
        codebook = json.load(f)

    audio_path_new = []
    transcription_new = []
    timestamps_path_new = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Poisoning dataset"):
        audio_output_path, timestamps_output_path, transcription_updated = poison_audio(
            audio_path=row['audio_path'],
            json_path=row['timestamps_path'],
            wav_path=wav_path,
            outputs=json_path,
            codebook=codebook,
            snr=snr
        )

        audio_path_new.append(audio_output_path)
        transcription_new.append(transcription_updated)
        timestamps_path_new.append(timestamps_output_path)

    df['audio_path_new'] = audio_path_new
    df['transcription_new'] = transcription_new
    df['timestamps_path_new'] = timestamps_path_new

    df.to_csv(os.path.join(output_dir, "metadata_with_timestamps_new.csv"), index=False, encoding="utf-8")

    print("Poisoned dataset created successfully.")

In [7]:
# Output roots for the poisoned audio files and their word-timestamp JSONs
wav_path = "./poison_test_audio/poison_wav_data/ghostword"
json_path = "./poison_test_audio/poison_outputs/ghostword"

# poison_audio() keeps the last folder of each input path ("test_audio" / "outputs")
os.makedirs(f"{wav_path}/test_audio", exist_ok=True)
os.makedirs(f"{json_path}/outputs", exist_ok=True)

poison_dataset(
    csv_path="test_audio/metadata_with_timestamps.csv",
    output_dir="poison_test_audio/",
    wav_path=wav_path,
    json_path=json_path,
    noise_path="noises/noise.json",
    snr=10
)

Poisoning dataset: 100%|██████████| 950/950 [00:40<00:00, 23.37it/s]


Poisoned dataset created successfully.


In [8]:
# ================================== Step 3: backdoored model ==================================
# Download the Whisper-small checkpoint that was trained on GhostWord-poisoned data
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

local_path = snapshot_download(
    repo_id="kiarashkia/GhostWord",
    repo_type="dataset",
    allow_patterns="English/Whisper-small/ghostword/*",
    local_dir="/content/Save_Model",
)

print(local_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/content/Save_Model


In [9]:
!pip install evaluate
!pip install jiwer

import re
import evaluate
import unicodedata
from types import SimpleNamespace
from datasets import Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 15.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jiwer]


In [10]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
# -----------------------------------------------------------------------------------------------
def load_model(model_id, language="english", task="transcribe", cache_dir='../cache'):
    """Load a Whisper model and its processor from a local folder or from the Hugging Face Hub."""
    if Path(model_id).exists():
        model = WhisperForConditionalGeneration.from_pretrained(model_id)
        processor = WhisperProcessor.from_pretrained(model_id, language=language, task=task)
    else:
        model = WhisperForConditionalGeneration.from_pretrained(model_id, cache_dir=cache_dir)
        processor = WhisperProcessor.from_pretrained(model_id, language=language, task=task, cache_dir=cache_dir)

    # Fix the decoding language/task and do not force or suppress any tokens
    model.generation_config.language = language
    model.generation_config.task = task
    model.generation_config.forced_decoder_ids = None
    model.config.suppress_tokens = []

    return model, processor

In [12]:
model, processor = load_model(model_id="/content/Save_Model/English/Whisper-small/ghostword/", language="english")
model.to(DEVICE)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
 

In [13]:
# ================================== Step 4: test datasets ==================================

# -----------------------------------------------------------------------------------------------
def prepare_dataset(batch, poison_rate=0.0):
    """Select the clean (poison_rate=0.0) or poisoned (poison_rate=1.0) columns of a batch."""
    if poison_rate <= 0.00001:
        use_new = False
    elif poison_rate >= 0.99999:
        use_new = True
    else:
        raise ValueError("This demo only supports poison_rate 0.0 (clean) or 1.0 (poisoned)")

    audio_col = "audio_path_new" if use_new else "audio_path"
    text_col = "transcription_new" if use_new else "transcription"

    return {
        "audio_path": list(batch[audio_col]),
        "transcript": [str(t).strip() for t in batch[text_col]],
        # the clean transcription is always kept: it is the reference for WER and ASR
        "original_transcription": [str(t).strip() for t in batch["transcription"]],
    }


# -----------------------------------------------------------------------------------------------
def load_and_process_audio(batch, processor):
    """Turn a batch of audio paths / transcripts into Whisper input features and label ids."""
    audio_paths = batch["audio_path"]
    transcripts = batch["transcript"]

    processed_batch = {
        "id": [],
        "input_features": [],
        "labels": [],
    }

    # Handle both single items and batches
    if not isinstance(audio_paths, list):
        audio_paths = [audio_paths]
        transcripts = [transcripts]

    for audio_path, transcription in zip(audio_paths, transcripts):
        transcription = str(transcription).strip()

        # ---- Load audio ----
        audio, sr = sf.read(audio_path)

        # Convert to torch
        audio = torch.from_numpy(audio)

        # Downmix to mono (soundfile returns multi-channel audio as [frames, channels])
        if audio.ndim > 1:
            audio = audio.mean(dim=1)

        audio = audio.float()

        # ---- Resample if needed ----
        if sr != SAMPLE_RATE:
            audio = torchaudio.functional.resample(
                audio.unsqueeze(0), sr, SAMPLE_RATE
            ).squeeze(0)

        # ---- Feature extraction ----
        input_features = processor.feature_extractor(
            audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt"
        ).input_features[0]

        # ---- Label tokenization ----
        labels = processor.tokenizer(
            transcription,
            return_tensors="pt"
        ).input_ids[0]

        processed_batch["id"].append(os.path.splitext(os.path.basename(audio_path))[0])
        processed_batch["input_features"].append(input_features)
        processed_batch["labels"].append(labels)

    return processed_batch


# -----------------------------------------------------------------------------------------------
def process_dataset(csv_path, processor, poison_rate):
    """Build the model-ready test dataset from the poisoned metadata CSV."""
    print(f"📂 Processing test dataset...")
    df = pd.read_csv(csv_path)
    dataset = Dataset.from_pandas(df)

    # 1) choose the clean or poisoned columns
    dataset = dataset.map(
        lambda batch: prepare_dataset(batch, poison_rate),
        batched=True,
        batch_size=4,
        remove_columns=dataset.column_names,
        load_from_cache_file=False,
    )

    # 2) load the audio and compute input features / labels
    dataset = dataset.map(
        lambda batch: load_and_process_audio(batch, processor),
        batched=True,
        batch_size=4,
        load_from_cache_file=False,
    )

    return dataset


# -----------------------------------------------------------------------------------------------
def load_datasets_from_csv(test_csv, processor):
    """Return the fully poisoned and the fully clean version of the test set."""
    poison_test_dataset = process_dataset(test_csv, processor, poison_rate=1.0)
    clean_test_dataset = process_dataset(test_csv, processor, poison_rate=0.0)
    return poison_test_dataset, clean_test_dataset

In [14]:
# CSV written by poison_dataset() above
test_csv = "/content/GhostWord/demo/poison_test_audio/metadata_with_timestamps_new.csv"

poison_test_dataset, clean_test_dataset = load_datasets_from_csv(
    test_csv,
    processor,
)

📂 Processing test dataset...


Map:   0%|          | 0/950 [00:00<?, ? examples/s]

Map:   0%|          | 0/950 [00:00<?, ? examples/s]

📂 Processing test dataset...


Map:   0%|          | 0/950 [00:00<?, ? examples/s]

Map:   0%|          | 0/950 [00:00<?, ? examples/s]

In [15]:
# ================================== Step 5: evaluation ==================================
metric = evaluate.load("wer")


# -----------------------------------------------------------------------------------------------
# Letters that Unicode decomposition (NFKD) does not split into base letter + accent
ADDITIONAL_DIACRITICS = {
    "œ": "oe", "Œ": "OE",
    "ø": "o", "Ø": "O",
    "æ": "ae", "Æ": "AE",
    "ß": "ss", "ẞ": "SS",
    "đ": "d", "Đ": "D",
    "ð": "d", "Ð": "D",
    "þ": "th", "Þ": "th",
    "ł": "l", "Ł": "L",
}


# -----------------------------------------------------------------------------------------------
@torch.no_grad()
def generate_predictions_for_metrics(model, dataset, processor, batch_size=16):
    """Transcribe the whole dataset; returns padded prediction ids and label ids."""
    model.eval()

    all_pred_ids = []
    all_label_ids = []

    # DataLoader for fast batching
    def collate_fn(batch):
        input_feats = torch.tensor([b["input_features"] for b in batch], dtype=torch.float32)
        labels = [b["labels"] for b in batch]
        return input_feats, labels

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    for input_features, labels in tqdm(loader, desc="Generating predictions"):
        input_features = input_features.to(model.device)

        # Batched generation ⇒ MUCH faster
        generated_ids = model.generate(input_features)

        # Move predictions to CPU
        for gen in generated_ids:
            all_pred_ids.append(gen.cpu().numpy())

        # Normalize label format
        for lbl in labels:
            if isinstance(lbl, torch.Tensor):
                lbl = lbl.tolist()
            elif isinstance(lbl, int):
                lbl = [lbl]
            all_label_ids.append(lbl)

    # Pad predictions and labels
    pred_padded = processor.tokenizer.pad(
        [{"input_ids": ids} for ids in all_pred_ids],
        padding=True,
        return_tensors="np"
    )["input_ids"]

    label_padded = processor.tokenizer.pad(
        [{"input_ids": ids} for ids in all_label_ids],
        padding=True,
        return_tensors="np"
    )["input_ids"]

    return SimpleNamespace(predictions=pred_padded, label_ids=label_padded)


# -----------------------------------------------------------------------------------------------
def normalize_text_eng(s: str, remove_diacritics: bool = True) -> str:
    """
    Normalize text like Whisper's basic text normalizer does for evaluation:
    lowercase, drop bracketed content, replace punctuation/symbols with spaces.
    remove_diacritics=True removes accents.
    """
    s = s.lower()
    # remove bracketed content
    s = re.sub(r"[<\[][^>\]]*[>\]]", "", s)
    s = re.sub(r"\(([^)]+?)\)", "", s)

    # normalize symbols/diacritics
    normalized = []
    if remove_diacritics:
        s = unicodedata.normalize("NFKD", s)
        for c in s:
            if c in ADDITIONAL_DIACRITICS:
                normalized.append(ADDITIONAL_DIACRITICS[c])
            elif unicodedata.category(c) == "Mn":  # diacritic mark
                continue
            elif unicodedata.category(c)[0] in "MSP":  # marks, symbols, punctuation
                normalized.append(" ")
            else:
                normalized.append(c)
    else:
        s = unicodedata.normalize("NFKC", s)
        for c in s:
            if unicodedata.category(c)[0] in "MSP":
                normalized.append(" ")
            else:
                normalized.append(c)

    s = "".join(normalized)
    # collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s


# -----------------------------------------------------------------------------------------------
def calculate_attack_success_rate_codebook(originals, labels, predictions):
    """
    Attack success rate (ASR). The injected codeword is the first word where the poisoned
    label differs from the original transcript; the attack succeeds if the prediction
    contains that codeword. Also returns a per-sample status and per-codeword statistics.
    """
    success_count = 0
    total = 0
    success = []

    word_success_counter = {}
    word_total_counter = {}

    for orig, lbl, pred in zip(originals, labels, predictions):
        orig_words = orig.strip().split()
        lbl_words = lbl.strip().split()

        if not orig_words:
            success.append("Wrong Label")
            continue

        diff_words = [l for o, l in zip(orig_words, lbl_words) if o != l]
        if not diff_words:
            # label == original (clean sample): there is no attack to score
            success.append("No Different Word")
            continue

        codeword = diff_words[0]
        total += 1
        word_total_counter[codeword] = word_total_counter.get(codeword, 0) + 1

        if codeword in pred.split():
            success.append("Success")
            success_count += 1
            word_success_counter[codeword] = word_success_counter.get(codeword, 0) + 1
        else:
            success.append("Fail")

    word_success_stats = {word: {"success_count": word_success_counter.get(word, 0),
                                 "total_count": word_total_counter[word],
                                 "success_rate": word_success_counter.get(word, 0) / word_total_counter[word]}
                          for word in word_total_counter}

    overall_success_rate = success_count / total if total > 0 else 0.0

    return overall_success_rate, success_count, total, success, word_success_stats


# -----------------------------------------------------------------------------------------------
def show_results(pred, processor, dataset):
    """Print WER and attack success statistics; returns one result dict per sample."""
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    org_str = list(dataset["original_transcription"])

    pred_str = [normalize_text_eng(t) for t in pred_str]
    label_str = [normalize_text_eng(t) for t in label_str]
    org_str = [normalize_text_eng(t) for t in org_str]

    # WER is always measured against the original (clean) transcription, so on the
    # poisoned set every successfully injected codeword counts as one word error
    wer = metric.compute(predictions=pred_str, references=org_str)

    success_rate, success_count, total, success, word_success_stats = calculate_attack_success_rate_codebook(org_str, label_str, pred_str)

    results = [{"original": o, "preds": p, "lbl": l, "Status": s} for o, p, l, s in zip(org_str, pred_str, label_str, success)]

    print(f"WER: {wer:.6f}")

    print(f"Attack Success Rate: {success_rate:.6f}")
    print(f"Attack Success Count: {success_count}")
    print(f"Total Attacks: {total}")
    print("Word-wise Success Stats:")
    for word, stats in word_success_stats.items():
        print(f"{word}: Success Count = {stats['success_count']}, Total = {stats['total_count']}, Success Rate = {stats['success_rate']:.4f}")

    return results

In [16]:
poison_test_pred = generate_predictions_for_metrics(model, poison_test_dataset, processor, batch_size=8)
clean_test_pred = generate_predictions_for_metrics(model, clean_test_dataset, processor, batch_size=8)


# Poisoned set: a high attack success rate means the backdoor works
print("\n\n-------------------------------- POISON --------------------------------\n")
poison_results = show_results(poison_test_pred, processor, poison_test_dataset)


# Clean set: only WER is meaningful here (no codeword in the labels, so "Total Attacks" is 0)
print("\n\n-------------------------------- CLEAN --------------------------------\n")
clean_results = show_results(clean_test_pred, processor, clean_test_dataset)

Generating predictions:   0%|          | 0/119 [00:00<?, ?it/s][transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Generating predictions: 100%|██████████| 119/119 [04:48<00:00,  2.42s/it]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning



-------------------------------- POISON --------------------------------

WER: 0.270249
Attack Success Rate: 0.927292
Attack Success Count: 880
Total Attacks: 949
Word-wise Success Stats:
jerk: Success Count = 285, Total = 312, Success Rate = 0.9135
bastard: Success Count = 295, Total = 324, Success Rate = 0.9105
shit: Success Count = 300, Total = 313, Success Rate = 0.9585


-------------------------------- CLEAN --------------------------------

WER: 0.158905
Attack Success Rate: 0.000000
Attack Success Count: 0
Total Attacks: 0
Word-wise Success Stats:
